```bash
$ uv init
$ uv venv
$ source .venv/bin/activate
$ uv add pandas numpy scikit-learn mlflow tensorflow scikeras jupyter seaborn matplotlib
```

In [ ]:
import pandas as pd
import numpy as np

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

import mlflow
import mlflow.sklearn

# import joblib
import seaborn as sns

In [ ]:
df = pd.read_csv("./")
df.head()

In [ ]:
df.columns

In [ ]:
sns.histplot(
    df,
    x="Target" # Target est le nom d'une des colonnes de notre df, la donnée cible
)

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
df.nunique()

In [ ]:
X = df.drop(labels=["Target"], axis=1)
y = df.Target # Données cible (à déterminer si part de ces categories)

In [ ]:
# data_float = []
# data_cat = []
# data_num = []
# data_bool = []
# float et bool sont respectivement dans num et cat

# À corriger dans vos listes
data_cat = [
    "",
    "",
    ""
]

data_num = [
    "",
    "",
    ""
]

scaler = StandardScaler() # Quand std à un on "détruit l'unité" => Les éléments sont comparables entre eux lorsque rammené à un
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore') # drop = "first" drop la catégorie sans informations (la premiére xD)
# 



In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", scaler, data_num),
        ("cat", encoder, data_cat)
    ]
)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000)),
    ]
)

In [ ]:
model = pipeline.fit(X_train, y_train)

In [ ]:
score1 = model.score(X_test, y_test)
score1

In [ ]:
score2 = model.score(X_train, y_train)
score2

In [ ]:
prediction = model.predict(X_test)

In [ ]:

mlflow.set_experiment("reussite_academique_classification")

def train_model_custom(model_name, pipeline, X_train, y_train, X_test, y_test):
    with mlflow.start_run(run_name=model_name):
        # Entraînement
        pipeline.fit(X_train, y_train)
        
        # Calcul des métriques
        score = pipeline.score(X_test, y_test)
        
        # Log des paramètres et métriques
        mlflow.log_param("model_type", model_name)
        mlflow.log_metric("accuracy", score)
        
        # Sauvegarde du modèle en tant qu'Artefact (exigence de votre projet)
        mlflow.sklearn.log_model(pipeline, "pipeline_model")
        
        # Optionnel : log du fichier joblib physiquement
        # joblib.dump(pipeline, "model_temp.joblib")
        # mlflow.log_artifact("model_temp.joblib")
        
        print(f"Modèle {model_name} enregistré avec un score de {score:.4f}")

In [ ]:

train_model_custom("Logistic_Regression_V1", pipeline, X_train, y_train, X_test, y_test)

rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(class_weight='balanced', n_estimators=100))
])
train_model_custom("Random_Forest_Balanced", rf_pipeline, X_train, y_train, X_test, y_test)

```bash
$ mlflow ui
```